# 🐾 Dog Emotion Classification


## Deep Learning-Based Multi-Class Image Classification

---

> **Project Overview**  
> This notebook presents a complete pipeline for classifying dog emotions into four categories: **Angry 😠**, **Happy 😊**, **Sad 😢**, and **Relaxed 😌** using two lightweight yet powerful CNN architectures — **MobileNetV2** and **EfficientNetB0**. Both models are optimized for speed and accuracy, making them ideal for real-time inference.

| Field | Detail |
|---|---|
| **Task** | Multi-class Image Classification |
| **Classes** | Angry, Happy, Sad, Relaxed |
| **Models** | MobileNetV2, EfficientNetB0 |
| **Framework** | TensorFlow / Keras |
| **Image Format** | JPG / JPEG |
| **Strategy** | Transfer Learning + Fine-Tuning |

---

## 1. 📦 Environment Setup & Library Imports

We begin by importing all the necessary libraries. This includes:
- **TensorFlow/Keras** — for building and training deep learning models
- **NumPy & Pandas** — for data manipulation
- **Matplotlib & Seaborn** — for rich visualizations
- **Scikit-learn** — for evaluation metrics (classification report, confusion matrix)
- **OS / Pathlib** — for filesystem operations

In [ ]:
import os
import warnings
import random
from pathlib import Path
from collections import Counter

warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib.patches import FancyBboxPatch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,accuracy_score, f1_score)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch    : {torch.__version__}')
print(f'Device     : {device}')
print('✅ All libraries loaded successfully.')

## 2. 📂 Dataset Loading & Path Configuration

We define all paths here in one place so the notebook stays easy to maintain. The dataset is organized in two ways:
- **Folder-based** — images stored in per-class subdirectories (`angry/`, `happy/`, `sad/`, `relaxed/`)
- **CSV-based** — `labels.csv` maps filenames to their emotion labels

We load both representations and verify that all referenced images exist on disk.

In [ ]:
BASE_DIR   = Path('/kaggle/input/datasets/maulikgajera/dog-emotion/Dog Emotion')
CSV_PATH   = Path('/kaggle/input/datasets/maulikgajera/dog-emotion/Dog Emotion/labels.csv')
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS     = 20
NUM_CLASSES = 4
CLASS_NAMES = ['angry', 'happy', 'relaxed', 'sad']
CLASS_COLORS = {
    'angry'  : '#E74C3C',
    'happy'  : '#2ECC71',
    'relaxed': '#3498DB',
    'sad'    : '#9B59B6'
}
CLASS_EMOJIS = {'angry': '😠', 'happy': '😊', 'relaxed': '😌', 'sad': '😢'}

df = pd.read_csv(CSV_PATH, index_col=0)
df.columns = ['filename', 'label']
df['label'] = df['label'].str.strip().str.lower()

df['filepath'] = df.apply(
    lambda r: BASE_DIR / r['label'] / r['filename'], axis=1
)

df['exists'] = df['filepath'].apply(lambda p: p.exists())
missing = (~df['exists']).sum()
df = df[df['exists']].reset_index(drop=True)

label2idx = {c: i for i, c in enumerate(sorted(df['label'].unique()))}
idx2label = {v: k for k, v in label2idx.items()}
df['label_idx'] = df['label'].map(label2idx)

print(f'Total samples loaded : {len(df):,}')
print(f'Missing / skipped    : {missing}')
print(f'Classes              : {list(label2idx.keys())}')
df.head()

## 3. 📊 Dataset Summary & Statistics

Before building any model, it's critical to understand the data. Here we produce:
- A **summary table** with per-class counts, percentages, and balance ratio
- Detection of any class imbalance that might require weighted loss or oversampling

In [ ]:
summary = (
    df.groupby('label')
      .size()
      .rename('count')
      .reset_index()
)
summary['percentage (%)'] = (summary['count'] / len(df) * 100).round(2)
summary['emoji']          = summary['label'].map(CLASS_EMOJIS)
summary['balance_ratio']  = (summary['count'] / summary['count'].max()).round(3)
summary = summary[['emoji', 'label', 'count', 'percentage (%)', 'balance_ratio']]
summary.columns = ['Emoji', 'Emotion', 'Image Count', 'Share (%)', 'Balance Ratio']

total_row = pd.DataFrame([{
    'Emoji': '📦', 'Emotion': 'TOTAL',
    'Image Count': summary['Image Count'].sum(),
    'Share (%)': 100.0,
    'Balance Ratio': '—'
}])
display_df = pd.concat([summary, total_row], ignore_index=True)

print('=' * 55)
print('          🐾  DOG EMOTION DATASET SUMMARY TABLE  🐾')
print('=' * 55)
print(display_df.to_string(index=False))
print('=' * 55)

min_r = summary['Balance Ratio'].astype(float).min()
if min_r < 0.8:
    print(f'⚠️  Class imbalance detected (min ratio = {min_r}). Consider class weights.')
else:
    print('✅ Dataset is well-balanced (all ratios ≥ 0.80).')

## 4. 📈 Data Visualization

Visual inspection of the label distribution helps us detect bias before training. We produce four complementary charts:
1. **Bar chart** — absolute count per class
2. **Donut / Pie chart** — proportional share
3. **Horizontal lollipop** — quick visual balance check
4. **Sample image grid** — 4 random images per class

In [ ]:
counts  = df['label'].value_counts().sort_index()
colors  = [CLASS_COLORS[c] for c in counts.index]

fig = plt.figure(figsize=(18, 6))
fig.suptitle('Dog Emotion Dataset — Label Distribution', fontsize=16,
             fontweight='bold', y=1.02)
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

ax1 = fig.add_subplot(gs[0])
bars = ax1.bar(counts.index, counts.values, color=colors,
               edgecolor='white', linewidth=1.5, width=0.6)
for bar, val in zip(bars, counts.values):
    ax1.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 10, f'{val:,}',
             ha='center', va='bottom', fontsize=10, fontweight='bold')
ax1.set_title('Image Count per Class', fontweight='bold')
ax1.set_ylabel('Number of Images')
ax1.set_xlabel('Emotion')
ax1.set_ylim(0, counts.max() * 1.15)
ax1.tick_params(axis='x', rotation=15)

ax2 = fig.add_subplot(gs[1])
wedges, texts, autotexts = ax2.pie(
    counts.values,
    labels=[f"{CLASS_EMOJIS[c]} {c}" for c in counts.index],
    colors=colors,
    autopct='%1.1f%%',
    startangle=140,
    pctdistance=0.78,
    wedgeprops=dict(width=0.55, edgecolor='white', linewidth=2)
)
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
ax2.set_title('Class Distribution (Donut)', fontweight='bold')
ax2.text(0, 0, f'{len(df):,}\nimages', ha='center', va='center',
         fontsize=11, fontweight='bold', color='#2C3E50')

ax3 = fig.add_subplot(gs[2])
y_pos = range(len(counts))
ax3.hlines(y_pos, 0, counts.values, colors=colors, linewidth=3, alpha=0.7)
ax3.scatter(counts.values, y_pos, color=colors, s=120, zorder=5)
ax3.set_yticks(list(y_pos))
ax3.set_yticklabels(
    [f"{CLASS_EMOJIS[c]} {c.capitalize()}" for c in counts.index],
    fontsize=10
)
for i, val in enumerate(counts.values):
    ax3.text(val + 5, i, str(val), va='center', fontsize=9, fontweight='bold')
ax3.set_title('Lollipop — Balance Check', fontweight='bold')
ax3.set_xlabel('Count')
ax3.set_xlim(0, counts.max() * 1.12)

plt.tight_layout()
plt.savefig('distribution_charts.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Distribution charts saved.')

### 4.1 Sample Image Grid

We display **4 random images per emotion class** to visually confirm the data quality and get an intuition for the visual features that distinguish each emotion.

In [ ]:
N_SAMPLES = 4
fig, axes = plt.subplots(NUM_CLASSES, N_SAMPLES,
                         figsize=(N_SAMPLES * 3.2, NUM_CLASSES * 3.2))
fig.suptitle('🐶 Sample Images per Emotion Class', fontsize=16,
             fontweight='bold', y=1.01)

for row_idx, emotion in enumerate(sorted(CLASS_NAMES)):
    subset = df[df['label'] == emotion].sample(n=N_SAMPLES, random_state=SEED)
    color  = CLASS_COLORS[emotion]
    for col_idx, (_, row) in enumerate(subset.iterrows()):
        ax = axes[row_idx][col_idx]
        try:
            img = Image.open(str(row['filepath'])).convert('RGB')
            img_resized = img.resize(IMG_SIZE)
            ax.imshow(img_resized)
        except Exception:
            ax.text(0.5, 0.5, 'N/A', ha='center', va='center',
                    transform=ax.transAxes)
        ax.axis('off')
        if col_idx == 0:
            ax.set_ylabel(
                f"{CLASS_EMOJIS[emotion]} {emotion.upper()}",
                fontsize=11, fontweight='bold', color=color,
                rotation=0, labelpad=60, va='center'
            )
        for spine in ['top','bottom','left','right']:
            ax.spines[spine].set_visible(True)
            ax.spines[spine].set_color(color)
            ax.spines[spine].set_linewidth(2.5)

plt.tight_layout()
plt.savefig('sample_grid.png', bbox_inches='tight', dpi=150)
plt.show()

## 5. 🔧 Data Preprocessing & Augmentation

We split the dataset into **train (70%) / validation (15%) / test (15%)** sets using stratified sampling to preserve class ratios across splits.

Data augmentation is applied **only on the training set** to improve generalization:
- Random horizontal flip
- Random rotation (±15°)
- Random zoom (±10%)
- Random brightness & contrast shifts

Both models expect pixel values in the range `[-1, 1]` (via `preprocess_input`).

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)

print(f'Train : {len(train_df):,}  ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df):,}   ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df):,}   ({len(test_df)/len(df)*100:.1f}%)')

class DogEmotionDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(str(row['filepath'])).convert('RGB')
        label = int(row['label_idx'])
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                        std=[0.229, 0.224, 0.225])
])

train_dataset = DogEmotionDataset(train_df, transform=train_transform)
val_dataset = DogEmotionDataset(val_df, transform=val_transform)
test_dataset = DogEmotionDataset(test_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print('✅ Dataset pipeline functions defined.')

### 5.1 Augmentation Preview

Visualizing augmented samples ensures our transformations look natural and don't distort the images to the point where the emotion cues are lost.

In [ ]:
sample_row = train_df[train_df['label'] == 'happy'].iloc[0]
sample_path = str(sample_row['filepath'])

def augment_once(path):
    img = Image.open(path).convert('RGB')
    transform = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.Resize(IMG_SIZE),
        transforms.ToTensor(),
    ])
    return (transform(img).permute(1, 2, 0) * 255).numpy().astype('uint8')

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('Data Augmentation Preview (Happy class)',
             fontsize=14, fontweight='bold')
orig = Image.open(sample_path).convert('RGB').resize(IMG_SIZE)
axes[0][0].imshow(orig)
axes[0][0].set_title('Original', fontweight='bold', color='#2ECC71')
axes[0][0].axis('off')
for i, ax in enumerate(axes.flat[1:]):
    ax.imshow(augment_once(sample_path))
    ax.set_title(f'Aug #{i+1}', fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.savefig('augmentation_preview.png', bbox_inches='tight', dpi=150)
plt.show()

## 6. 🚀 Model 1 — MobileNetV2

**MobileNetV2** is an efficient architecture by Google that uses **inverted residual blocks** with linear bottlenecks. Key advantages:
- ✅ ~3.4M parameters (very fast inference)
- ✅ Excellent accuracy-to-speed ratio on mobile/edge devices
- ✅ Pre-trained on ImageNet (1.4M images, 1000 classes)

**Strategy:** We freeze the base model, add a custom classification head, train it for a few epochs, then fine-tune the last 30 layers of the backbone for higher accuracy.

In [ ]:
class DogEmotionModel(nn.Module):
    def __init__(self, base_model, num_classes=NUM_CLASSES, dropout_rate=0.4):
        super(DogEmotionModel, self).__init__()
        self.base = base_model
        in_features = base_model.classifier[-1].in_features
        
        self.base.classifier = nn.Identity()
        
        self.head = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        x = self.base(x)
        x = self.head(x)
        return x

base_mob = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V2)
model_mob = DogEmotionModel(base_mob, num_classes=NUM_CLASSES)
model_mob = model_mob.to(device)

for param in model_mob.base.parameters():
    param.requires_grad = False

print("MobileNetV2 model created and moved to device")

### 6.1 MobileNetV2 — Phase 1: Head Training

We first train only the custom classification head (backbone frozen) for **10 epochs**. This avoids destroying the pre-trained weights with large gradients early in training.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_mob.parameters(), lr=1e-3)

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)
    
    return total_loss / len(loader), correct / total

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total += labels.size(0)
    
    return total_loss / len(loader), correct / total

print('🏋️  Phase 1 — Training classification head (backbone frozen)...')
history_mob1 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_acc = 0
patience_counter = 0

for epoch in range(10):
    train_loss, train_acc = train_epoch(model_mob, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model_mob, val_loader, criterion, device)
    
    history_mob1['loss'].append(train_loss)
    history_mob1['accuracy'].append(train_acc)
    history_mob1['val_loss'].append(val_loss)
    history_mob1['val_accuracy'].append(val_acc)
    
    print(f'Epoch {epoch+1}/10 | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_mob.state_dict(), 'best_mobilenetv2.pth')
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'✅ Phase 1 complete. Best val_accuracy: {max(history_mob1["val_accuracy"]):.4f}')

### 6.2 MobileNetV2 — Phase 2: Fine-Tuning

With the head stabilized, we **unfreeze the last 30 layers** of the backbone and continue training with a much lower learning rate (`1e-5`) to gently adapt the deep features to our dog emotion domain.

In [ ]:
for i, param in enumerate(model_mob.base.parameters()):
    if i < len(list(model_mob.base.parameters())) - 30:
        param.requires_grad = False
    else:
        param.requires_grad = True

optimizer = optim.Adam([p for p in model_mob.parameters() if p.requires_grad], lr=1e-5)

print('🔬  Phase 2 — Fine-tuning last 30 backbone layers...')
history_mob2 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_acc = max(history_mob1['val_accuracy'])
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model_mob, train_loader, criterion, optimizer, device)
    val_loss, val_acc = validate(model_mob, val_loader, criterion, device)
    
    history_mob2['loss'].append(train_loss)
    history_mob2['accuracy'].append(train_acc)
    history_mob2['val_loss'].append(val_loss)
    history_mob2['val_accuracy'].append(val_acc)
    
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Val Acc: {val_acc:.4f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_mob.state_dict(), 'best_mobilenetv2.pth')
    else:
        patience_counter += 1
        if patience_counter >= 6:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'✅ Fine-tuning complete. Best val_accuracy: {max(history_mob2["val_accuracy"]):.4f}')

## 7. ⚡ Model 2 — EfficientNetB0

**EfficientNetB0** is the smallest variant of the EfficientNet family, which uses a principled **compound scaling** approach (simultaneously scaling depth, width, and resolution). Key advantages:
- ✅ ~5.3M parameters — still very fast
- ✅ Top-1 accuracy on ImageNet significantly higher than MobileNetV2
- ✅ State-of-the-art performance for its FLOP budget

We follow the same two-phase training strategy as MobileNetV2.

In [ ]:
class EfficientNetHead(nn.Module):
    def __init__(self, in_features, num_classes=NUM_CLASSES):
        super(EfficientNetHead, self).__init__()
        self.net = nn.Sequential(
            nn.BatchNorm1d(in_features),
            nn.Linear(in_features, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(128, num_classes)
        )
    
    def forward(self, x):
        return self.net(x)

base_eff = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
in_features = base_eff.classifier[1].in_features
base_eff.classifier = nn.Identity()

model_eff = nn.Sequential(base_eff, EfficientNetHead(in_features, NUM_CLASSES))
model_eff = model_eff.to(device)

for param in base_eff.parameters():
    param.requires_grad = False

print("EfficientNetB0 model created and moved to device")

### 7.1 EfficientNetB0 — Phase 1: Head Training

In [ ]:
print('🏋️  Phase 1 — Training classification head (backbone frozen)...')
optimizer_eff = optim.Adam(model_eff.parameters(), lr=1e-3)
history_eff1 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_acc = 0
patience_counter = 0

for epoch in range(10):
    train_loss, train_acc = train_epoch(model_eff, train_loader, criterion, optimizer_eff, device)
    val_loss, val_acc = validate(model_eff, val_loader, criterion, device)
    
    history_eff1['loss'].append(train_loss)
    history_eff1['accuracy'].append(train_acc)
    history_eff1['val_loss'].append(val_loss)
    history_eff1['val_accuracy'].append(val_acc)
    
    print(f'Epoch {epoch+1}/10 | Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_eff.state_dict(), 'best_efficientnetb0.pth')
    else:
        patience_counter += 1
        if patience_counter >= 5:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'✅ Phase 1 complete. Best val_accuracy: {max(history_eff1["val_accuracy"]):.4f}')

### 7.2 EfficientNetB0 — Phase 2: Fine-Tuning

In [ ]:
for i, param in enumerate(base_eff.parameters()):
    if i < len(list(base_eff.parameters())) - 30:
        param.requires_grad = False
    else:
        param.requires_grad = True

optimizer_eff = optim.Adam([p for p in model_eff.parameters() if p.requires_grad], lr=1e-5)

print('🔬  Phase 2 — Fine-tuning last 30 backbone layers...')
history_eff2 = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
best_val_acc = max(history_eff1['val_accuracy'])
patience_counter = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model_eff, train_loader, criterion, optimizer_eff, device)
    val_loss, val_acc = validate(model_eff, val_loader, criterion, device)
    
    history_eff2['loss'].append(train_loss)
    history_eff2['accuracy'].append(train_acc)
    history_eff2['val_loss'].append(val_loss)
    history_eff2['val_accuracy'].append(val_acc)
    
    if (epoch + 1) % 2 == 0:
        print(f'Epoch {epoch+1}/{EPOCHS} | Val Acc: {val_acc:.4f}')
    
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_eff.state_dict(), 'best_efficientnetb0.pth')
    else:
        patience_counter += 1
        if patience_counter >= 6:
            print(f'Early stopping at epoch {epoch+1}')
            break

print(f'✅ Fine-tuning complete. Best val_accuracy: {max(history_eff2["val_accuracy"]):.4f}')

## 8. 📉 Training History Visualization

We merge Phase 1 and Phase 2 histories and plot **accuracy** and **loss** curves side by side for both models. A vertical dashed line marks the transition between frozen-head training and fine-tuning.

In [ ]:
def merge_histories(h1, h2):
    merged = {}
    for key in h1:
        merged[key] = h1[key] + h2[key]
    return merged

hist_mob = merge_histories(history_mob1, history_mob2)
hist_eff = merge_histories(history_eff1, history_eff2)
phase1_end = len(history_mob1['accuracy'])

def plot_training(hist, title, color, ax_acc, ax_loss):
    epochs_range = range(1, len(hist['accuracy']) + 1)
    ax_acc.plot(epochs_range, hist['accuracy'],     color=color,      lw=2,   label='Train Acc')
    ax_acc.plot(epochs_range, hist['val_accuracy'], color=color, lw=2, ls='--', label='Val Acc')
    ax_acc.axvline(phase1_end, ls=':', color='gray', lw=1.5, label='Fine-tune starts')
    ax_acc.set_title(f'{title}\nAccuracy', fontweight='bold')
    ax_acc.set_ylabel('Accuracy')
    ax_acc.set_xlabel('Epoch')
    ax_acc.legend(fontsize=8)
    ax_acc.set_ylim(0, 1.05)
    
    ax_loss.plot(epochs_range, hist['loss'],     color=color,      lw=2,   label='Train Loss')
    ax_loss.plot(epochs_range, hist['val_loss'], color=color, lw=2, ls='--', label='Val Loss')
    ax_loss.axvline(phase1_end, ls=':', color='gray', lw=1.5)
    ax_loss.set_title(f'{title}\nLoss', fontweight='bold')
    ax_loss.set_ylabel('Loss')
    ax_loss.set_xlabel('Epoch')
    ax_loss.legend(fontsize=8)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('📉 Training & Validation Curves', fontsize=16, fontweight='bold')
plot_training(hist_mob, 'MobileNetV2',  '#E74C3C', axes[0][0], axes[1][0])
plot_training(hist_eff, 'EfficientNetB0', '#3498DB', axes[0][1], axes[1][1])
plt.tight_layout()
plt.savefig('training_curves.png', bbox_inches='tight', dpi=150)
plt.show()

## 9. 🧪 Model Evaluation on Test Set

We evaluate both models on the held-out test set and collect predictions for the confusion matrix and classification report. Performance is measured using:
- **Accuracy** — overall correctness
- **Macro F1-score** — balanced metric across all classes
- **AUC** — area under the ROC curve (higher is better)

In [ ]:
def evaluate_model(model, test_loader, test_df, model_name, device):
    model.eval()
    y_pred_list = []
    y_true_list = []
    total_correct = 0
    total_samples = 0
    total_loss = 0
    
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total_correct += predicted.eq(labels).sum().item()
            total_samples += labels.size(0)
            
            y_pred_list.extend(predicted.cpu().numpy())
            y_true_list.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(test_loader)
    accuracy = total_correct / total_samples
    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)
    f1 = f1_score(y_true, y_pred, average='macro')
    
    from sklearn.metrics import roc_auc_score
    y_pred_proba = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            outputs = model(images)
            proba = torch.nn.functional.softmax(outputs, dim=1)
            y_pred_proba.extend(proba.cpu().numpy())
    y_pred_proba = np.array(y_pred_proba)
    auc = roc_auc_score(y_true, y_pred_proba, multi_class='ovr')

    print(f'\n{"─"*45}')
    print(f'  {model_name}')
    print(f'{"─"*45}')
    print(f'  Loss     : {avg_loss:.4f}')
    print(f'  Accuracy : {accuracy:.4f}  ({accuracy*100:.2f}%)')
    print(f'  AUC      : {auc:.4f}')
    print(f'  Macro F1 : {f1:.4f}')
    return dict(model=model_name, loss=avg_loss, accuracy=accuracy, auc=auc, f1=f1), y_true, y_pred

model_mob.load_state_dict(torch.load('best_mobilenetv2.pth'))
model_eff.load_state_dict(torch.load('best_efficientnetb0.pth'))

metrics_mob, y_true_mob, y_pred_mob = evaluate_model(
    model_mob, test_loader, test_df, 'MobileNetV2', device
)
metrics_eff, y_true_eff, y_pred_eff = evaluate_model(
    model_eff, test_loader, test_df, 'EfficientNetB0', device
)

## 10. 📊 Model Comparison Dashboard

A side-by-side radar/bar comparison of all key metrics helps us objectively decide which model to deploy. We also include a **grouped bar chart** for a high-level view.

In [ ]:
comparison_df = pd.DataFrame([metrics_mob, metrics_eff])
comparison_df = comparison_df.set_index('model')

metrics_to_plot = ['accuracy', 'auc', 'f1']
metric_labels   = ['Accuracy', 'AUC', 'Macro F1']
model_colors    = ['#E74C3C', '#3498DB']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🏆 Model Comparison — MobileNetV2 vs EfficientNetB0',
             fontsize=15, fontweight='bold')

x      = np.arange(len(metric_labels))
width  = 0.35
ax     = axes[0]
for i, (model_name, color) in enumerate(zip(comparison_df.index, model_colors)):
    vals = [comparison_df.loc[model_name, m] for m in metrics_to_plot]
    bars = ax.bar(x + i * width, vals, width, label=model_name,
                  color=color, alpha=0.85, edgecolor='white')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.005, f'{v:.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x + width / 2)
ax.set_xticklabels(metric_labels, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('Score')
ax.set_title('Metric Comparison', fontweight='bold')
ax.legend()

ax2 = fig.add_subplot(1, 2, 2, projection='polar')
categories = metric_labels + [metric_labels[0]]
N = len(metric_labels)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]
ax2.set_theta_offset(np.pi / 2)
ax2.set_theta_direction(-1)
ax2.set_thetagrids(np.degrees(angles[:-1]), metric_labels, fontsize=10)
for model_name, color in zip(comparison_df.index, model_colors):
    vals = [comparison_df.loc[model_name, m] for m in metrics_to_plot]
    vals += vals[:1]
    ax2.plot(angles, vals, 'o-', lw=2, color=color, label=model_name)
    ax2.fill(angles, vals, alpha=0.15, color=color)
ax2.set_ylim(0, 1)
ax2.set_title('Radar Chart', fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15))

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n📋 Comparison Table:')
print(comparison_df[metrics_to_plot + ['loss']].to_string())

## 11. 🔢 Confusion Matrices

The confusion matrix shows, for each true class, how many samples were correctly predicted vs. confused with another class. This reveals which emotion pairs are hardest to distinguish (e.g., *sad* vs *relaxed*).

In [ ]:
def plot_confusion(y_true, y_pred, class_names, title, ax, cmap='Blues'):
    cm = confusion_matrix(y_true, y_pred)
    cm_norm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
    labels = [f'{CLASS_EMOJIS[c]}\n{c.capitalize()}' for c in class_names]
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap=cmap,
                xticklabels=labels, yticklabels=labels,
                linewidths=0.5, linecolor='white',
                cbar_kws={'label': 'Proportion'},
                ax=ax, annot_kws={'size': 10})
    ax.set_title(title, fontweight='bold', fontsize=12, pad=12)
    ax.set_xlabel('Predicted Label', fontsize=10)
    ax.set_ylabel('True Label', fontsize=10)
    ax.tick_params(axis='both', labelsize=9)

sorted_classes = sorted(CLASS_NAMES)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Confusion Matrices (Test Set)', fontsize=15, fontweight='bold')
plot_confusion(y_true_mob, y_pred_mob, sorted_classes,
               'MobileNetV2', ax1, cmap='Reds')
plot_confusion(y_true_eff, y_pred_eff, sorted_classes,
               'EfficientNetB0', ax2, cmap='Blues')
plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight', dpi=150)
plt.show()

## 12. 📝 Classification Reports

The per-class **precision**, **recall**, and **F1-score** reveal any class-specific weaknesses. A well-performing model should have consistently high scores across all four emotion classes.

In [ ]:
sorted_classes = sorted(CLASS_NAMES)
target_names   = [f"{CLASS_EMOJIS[c]} {c.capitalize()}" for c in sorted_classes]

print('=' * 60)
print('       📋 MobileNetV2 — Classification Report')
print('=' * 60)
print(classification_report(
    y_true_mob, y_pred_mob, target_names=target_names, digits=4
))

print('=' * 60)
print('       📋 EfficientNetB0 — Classification Report')
print('=' * 60)
print(classification_report(
    y_true_eff, y_pred_eff, target_names=target_names, digits=4
))

### 12.1 Per-Class F1 Bar Chart

Visualizing per-class F1 scores for both models side by side makes it easy to spot which classes benefit most from each architecture.

In [ ]:
from sklearn.metrics import f1_score

f1_mob = f1_score(y_true_mob, y_pred_mob, average=None)
f1_eff = f1_score(y_true_eff, y_pred_eff, average=None)

x     = np.arange(NUM_CLASSES)
width = 0.35
labels = [f"{CLASS_EMOJIS[c]} {c.capitalize()}" for c in sorted_classes]

fig, ax = plt.subplots(figsize=(12, 5))
b1 = ax.bar(x - width/2, f1_mob, width, label='MobileNetV2',   color='#E74C3C', alpha=0.85)
b2 = ax.bar(x + width/2, f1_eff, width, label='EfficientNetB0', color='#3498DB', alpha=0.85)

for bars in [b1, b2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.01, f'{bar.get_height():.3f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=11)
ax.set_ylim(0, 1.12)
ax.set_ylabel('F1 Score')
ax.set_title('Per-Class F1 Score — Both Models', fontsize=14, fontweight='bold')
ax.legend()
ax.axhline(0.80, ls='--', color='gray', lw=1, label='0.80 threshold')
plt.tight_layout()
plt.savefig('per_class_f1.png', bbox_inches='tight', dpi=150)
plt.show()

## 13. 🔍 Prediction on Sample Test Images

We qualitatively inspect model predictions on a random batch of test images. Each image displays the **true label**, the **predicted label from each model**, and the **confidence score**. Correct predictions are highlighted in green, incorrect in red.

In [ ]:
N_DISPLAY = 12
sample_test = test_df.sample(n=N_DISPLAY, random_state=SEED).reset_index(drop=True)

fig, axes = plt.subplots(3, 4, figsize=(18, 14))
fig.suptitle('🔍 Model Predictions on Sample Test Images',
             fontsize=15, fontweight='bold', y=1.01)

for idx, (_, row) in enumerate(sample_test.iterrows()):
    ax = axes[idx // 4][idx % 4]
    try:
        orig_img = Image.open(str(row['filepath'])).convert('RGB')
        ax.imshow(orig_img)

        tensor_img = val_transform(orig_img).unsqueeze(0).to(device)
        
        with torch.no_grad():
            pred_mob = torch.nn.functional.softmax(model_mob(tensor_img), dim=1)[0]
            cls_mob = idx2label[pred_mob.argmax().item()]
            conf_mob = pred_mob.max().item()

            pred_eff = torch.nn.functional.softmax(model_eff(tensor_img), dim=1)[0]
            cls_eff = idx2label[pred_eff.argmax().item()]
            conf_eff = pred_eff.max().item()

        true_label = row['label']
        c_mob = '#2ECC71' if cls_mob == true_label else '#E74C3C'
        c_eff = '#2ECC71' if cls_eff == true_label else '#E74C3C'

        title = (f"True: {CLASS_EMOJIS[true_label]} {true_label}\n"
                 f"MobV2: {cls_mob} ({conf_mob:.0%}) | "
                 f"EffB0: {cls_eff} ({conf_eff:.0%})")
        ax.set_title(title, fontsize=7.5, pad=4)

        border_color = '#2ECC71' if (cls_mob == true_label and cls_eff == true_label) else '#E74C3C'
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_color(border_color)
            spine.set_linewidth(3)
    except Exception as e:
        ax.text(0.5, 0.5, 'Error', ha='center', va='center', transform=ax.transAxes)
    ax.axis('off')

plt.tight_layout()
plt.savefig('sample_predictions.png', bbox_inches='tight', dpi=150)
plt.show()

## 14. 🏁 Final Summary & Recommendations

We consolidate all metrics into a final comparison table and provide a recommendation based on the results.

In [ ]:
params_mob = sum(p.numel() for p in model_mob.parameters())
params_eff = sum(p.numel() for p in model_eff.parameters())

final_table = pd.DataFrame({
    'Model'          : ['MobileNetV2', 'EfficientNetB0'],
    'Parameters'     : [f'{params_mob/1e6:.2f}M', f'{params_eff/1e6:.2f}M'],
    'Test Accuracy'  : [f"{metrics_mob['accuracy']*100:.2f}%", f"{metrics_eff['accuracy']*100:.2f}%"],
    'Macro F1'       : [f"{metrics_mob['f1']:.4f}", f"{metrics_eff['f1']:.4f}"],
    'AUC'            : [f"{metrics_mob['auc']:.4f}", f"{metrics_eff['auc']:.4f}"],
    'Speed'          : ['⚡⚡⚡', '⚡⚡']
})

print('=' * 70)
print('         🏆  FINAL MODEL COMPARISON SUMMARY  🏆')
print('=' * 70)
print(final_table.to_string(index=False))
print('=' * 70)

winner = 'EfficientNetB0' if metrics_eff['accuracy'] >= metrics_mob['accuracy'] else 'MobileNetV2'
print(f"""
📌 RECOMMENDATION
─────────────────────────────────────────────────────
🏆 Best Overall Accuracy : {winner}
⚡ Fastest Inference     : MobileNetV2
🎯 Balanced Choice       : EfficientNetB0

→ For real-time / edge deployment → MobileNetV2
→ For maximum accuracy            → EfficientNetB0
─────────────────────────────────────────────────────
""")

## 15. 💾 Save Models & Outputs

Both trained models are saved in the modern Keras `.keras` format (recommended over `.h5` for TF 2.x). All generated charts are also saved as high-DPI PNG files for reporting.

In [ ]:
torch.save(model_mob.state_dict(), 'mobilenetv2_dog_emotion_final.pth')
torch.save(model_eff.state_dict(), 'efficientnetb0_dog_emotion_final.pth')

import json
with open('label_map.json', 'w') as f:
    json.dump(idx2label, f, indent=2)

print('📁 Saved files:')
for fname in [
    'mobilenetv2_dog_emotion_final.pth',
    'efficientnetb0_dog_emotion_final.pth',
    'label_map.json',
    'distribution_charts.png',
    'sample_grid.png',
    'augmentation_preview.png',
    'training_curves.png',
    'model_comparison.png',
    'confusion_matrices.png',
    'per_class_f1.png',
    'sample_predictions.png'
]:
    print(f'  ✅ {fname}')

print('\n🎉 Project complete!')